# Notebook 2: Feature Engineering
Build the complete multi-factor feature matrix: technical + macro + sentiment + inventory.

## 1. Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
print("Ready.")

## 2. Technical Indicators

In [ ]:
from src.ingestion.download_data import generate_synthetic_prices
from pathlib import Path

if not Path('../data/raw/brent_ohlcv.parquet').exists():
    generate_synthetic_prices()

import pandas as pd
brent = pd.read_parquet('../data/raw/brent_ohlcv.parquet')

from src.features.technical_indicators import (
    add_moving_averages, add_rsi, add_macd, add_bollinger_bands,
    add_atr, add_rate_of_change, add_historical_volatility
)

df = add_moving_averages(brent)
df = add_rsi(df)
df = add_macd(df)
df = add_bollinger_bands(df)
df = add_atr(df)
df = add_rate_of_change(df)
df = add_historical_volatility(df)

print(f"Technical features added: {len(df.columns) - len(brent.columns)}")
df[['close','rsi_14','macd_hist','bb_pct_b','atr_pct']].tail(5)

## 3. Visualise Key Technical Indicators

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
recent = df.iloc[-252:]

axes[0].plot(recent.index, recent['close'], color='#378ADD', linewidth=1.2, label='Brent')
axes[0].plot(recent.index, recent['sma_20'], '--', color='#EF9F27', linewidth=0.8, label='SMA20')
axes[0].plot(recent.index, recent['sma_50'], '--', color='#E24B4A', linewidth=0.8, label='SMA50')
axes[0].fill_between(recent.index, recent['bb_lower'], recent['bb_upper'],
                     alpha=0.1, color='#378ADD', label='Bollinger Bands')
axes[0].set_ylabel('Price ($/bbl)')
axes[0].legend(fontsize=8)
axes[0].set_title('Brent Price with Moving Averages & Bollinger Bands')

axes[1].plot(recent.index, recent['rsi_14'], color='#EF9F27', linewidth=1.2)
axes[1].axhline(70, color='#E24B4A', linewidth=0.8, linestyle='--', label='Overbought (70)')
axes[1].axhline(30, color='#1D9E75', linewidth=0.8, linestyle='--', label='Oversold (30)')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=8)
axes[1].set_title('RSI (14)')

axes[2].bar(recent.index, recent['macd_hist'],
            color=['#1D9E75' if v > 0 else '#E24B4A' for v in recent['macd_hist']],
            alpha=0.7, width=1)
axes[2].plot(recent.index, recent['macd_line'], color='#378ADD', linewidth=0.8)
axes[2].plot(recent.index, recent['macd_signal'], color='#EF9F27', linewidth=0.8)
axes[2].set_ylabel('MACD')
axes[2].set_title('MACD (12, 26, 9)')

axes[3].plot(recent.index, recent['hvol_20d']*100, color='#E24B4A', linewidth=1.0)
axes[3].set_ylabel('Vol (%)')
axes[3].set_title('20-Day Realised Volatility (Annualised)')

plt.tight_layout()
plt.savefig('../docs/images/technical_indicators.png', bbox_inches='tight')
plt.show()

## 4. Macro Features

In [ ]:
from src.features.macro_features import add_macro_features, add_calendar_features, add_lagged_features

df = add_macro_features(df)
df = add_calendar_features(df)
df = add_lagged_features(df)

macro_cols = [c for c in df.columns if c.startswith(('dxy','yield','gold','spx','eurusd'))]
print(f"Macro features: {len(macro_cols)}")
print(macro_cols[:10])

# Correlation: macro vs 5-day forward return
corr = df[macro_cols + ['target_5d_return']].dropna().corr()['target_5d_return'].drop('target_5d_return')
top_macro = corr.abs().nlargest(10)
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#1D9E75' if v > 0 else '#E24B4A' for v in corr[top_macro.index]]
corr[top_macro.index].sort_values().plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='#888', linewidth=0.8)
ax.set_title('Top Macro Features — Correlation with 5-Day Forward Return')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('../docs/images/macro_features.png', bbox_inches='tight')
plt.show()

## 5. EIA Inventory Features

In [ ]:
from src.features.inventory_features import add_inventory_features

df = add_inventory_features(df)
inv_cols = [c for c in df.columns if c.startswith('eia_')]
print(f"Inventory features: {len(inv_cols)}")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index[-200:], df['eia_crude_inventory_mb'].iloc[-200:],
             color='#378ADD', linewidth=1.2)
axes[0].set_title('EIA Crude Inventory (Million Barrels)')
axes[0].set_ylabel('Inventory (Mbbl)')

axes[1].bar(df.index[-200:], df['eia_weekly_change_mb'].iloc[-200:],
            color=['#1D9E75' if v < 0 else '#E24B4A' for v in df['eia_weekly_change_mb'].iloc[-200:]],
            alpha=0.8, width=1)
axes[1].set_title('Weekly EIA Change (Draw=Green, Build=Red)')
axes[1].set_ylabel('Change (Mbbl)')
axes[1].axhline(0, color='#888', linewidth=0.6)
plt.tight_layout()
plt.savefig('../docs/images/eia_inventory.png', bbox_inches='tight')
plt.show()

## 6. NLP Sentiment Features

In [ ]:
from src.features.sentiment_analyzer import add_sentiment_features, SAMPLE_HEADLINES, FinBERTSentimentAnalyzer

# Demo: lexicon scoring on sample headlines
analyzer = FinBERTSentimentAnalyzer()
results = analyzer.analyze_headlines([h for h, _ in SAMPLE_HEADLINES[:10]])

demo_df = pd.DataFrame({
    'Headline': [h[:60] + '...' for h, _ in SAMPLE_HEADLINES[:10]],
    'True Label': [l for _, l in SAMPLE_HEADLINES[:10]],
    'Score': [round(r['score'], 3) for r in results],
    'Label': [r['label'] for r in results],
})
print(demo_df.to_string(index=False))

# Add to main dataframe
df = add_sentiment_features(df)
sent_cols = [c for c in df.columns if 'sentiment' in c]
print(f"\nSentiment features added: {sent_cols}")

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(df.index[-200:], df['close'].iloc[-200:], color='#378ADD', linewidth=1.0)
axes[0].set_ylabel('Brent Price')

axes[1].plot(df.index[-200:], df['sentiment_score'].iloc[-200:],
             color='#888', linewidth=0.6, alpha=0.6)
axes[1].plot(df.index[-200:], df['sentiment_ma5'].iloc[-200:],
             color='#EF9F27', linewidth=1.2, label='MA5')
axes[1].axhline(0, color='#888', linewidth=0.6)
axes[1].fill_between(df.index[-200:], 0, df['sentiment_score'].iloc[-200:],
                     where=df['sentiment_score'].iloc[-200:] > 0,
                     alpha=0.3, color='#1D9E75', label='Positive')
axes[1].fill_between(df.index[-200:], 0, df['sentiment_score'].iloc[-200:],
                     where=df['sentiment_score'].iloc[-200:] < 0,
                     alpha=0.3, color='#E24B4A', label='Negative')
axes[1].set_ylabel('Sentiment Score')
axes[1].set_title('NLP Market Sentiment vs Brent Price')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig('../docs/images/sentiment_features.png', bbox_inches='tight')
plt.show()

## 7. Run Full Feature Pipeline

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '../src/features/feature_pipeline.py', '--instrument', 'brent'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[:500])

## 8. Feature Matrix Summary

In [ ]:
from src.features.feature_pipeline import load_features
data = load_features('brent')
train_df = data['train_df']
feature_cols = data['feature_cols']

print(f"Feature matrix: {train_df.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Train: {train_df.index[0].date()} → {train_df.index[-1].date()}")
print(f"\nTarget stats:")
for tc in data['target_cols'][:3]:
    vals = train_df[tc].dropna()
    print(f"  {tc:<30} mean={vals.mean():.5f}  pct_up={( vals>0).mean():.1%}")